In [3]:
import requests
import urllib3
from bs4 import BeautifulSoup

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

url = 'https://steamcommunity.com/market/search/render/?appid=730&category_730_Weapon%5B%5D=tag_weapon_ak47&count=10&start=0'
headers = {'User-Agent': 'Mozilla/5.0'}

response = requests.get(url, headers=headers, verify=False)
data = response.json()

print(f"Total results: {data['total_count']}")
print()

soup = BeautifulSoup(data['results_html'], 'html.parser')
rows = soup.find_all('a', class_='market_listing_row_link')

for row in rows:
    name = row.find('span', class_='market_listing_item_name').text
    qty = row.find('span', class_='market_listing_num_listings_qty').text.strip()
    price = row.find('span', class_='normal_price', attrs={'data-price': True}).text.strip()
    print(f"{name}: {price} | Listings: {qty}")

Total results: 570

AK-47 | Nightwish (Minimal Wear): $99.33 USD | Listings: 184
AK-47 | Point Disarray (Field-Tested): $17.80 USD | Listings: 352
AK-47 | Aphrodite (Battle-Scarred): $16.99 USD | Listings: 748
AK-47 | Aphrodite (Well-Worn): $17.33 USD | Listings: 796
AK-47 | Ice Coaled (Field-Tested): $5.56 USD | Listings: 2,226
AK-47 | Inheritance (Field-Tested): $56.07 USD | Listings: 751
AK-47 | Crane Flight (Minimal Wear): $33.75 USD | Listings: 586
AK-47 | Nightwish (Well-Worn): $82.00 USD | Listings: 198
AK-47 | Slate (Field-Tested): $5.69 USD | Listings: 4,126
AK-47 | Phantom Disruptor (Field-Tested): $6.94 USD | Listings: 1,281


In [5]:
import requests
import urllib3
import time
import csv
from datetime import datetime
from bs4 import BeautifulSoup   

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

BASE_URL = 'https://steamcommunity.com/market/search/render/'
APP_ID = 730
TAG = 'tag_weapon_ak47'
COUNT = 100
DELAY = 1.5
MAX_RETRIES = 3

headers = {'User-Agent': 'Mozilla/5.0'}


def fetch_page(start):
    
    params = {
        'appid': APP_ID,
        'category_730_Weapon[]': TAG,
        'count': COUNT,
        'start': start,
        'sort_column': 'name',
        'sort_dir': 'asc',
    }

    attempt = 1
    while attempt <= MAX_RETRIES:
        try:
            response = requests.get(BASE_URL, headers=headers, params=params, verify=False, timeout=10)
            response.raise_for_status()
            return response.json()
        except Exception as e:
            print(f"Versuch {attempt} fehlgeschlagen bei start={start}: {e}")
            attempt = attempt + 1
            time.sleep(DELAY)

    print(f"Aufgegeben bei start={start}")
    return None


def parse_page(html):
    soup = BeautifulSoup(html, 'html.parser')
    rows = soup.find_all('a', class_='market_listing_row_link')

    items = []
    for row in rows:
        div = row.find('div', class_='market_listing_row')

        hash_name = div.get('data-hash-name')
        url = row.get('href')

        qty_span = div.find('span', class_='market_listing_num_listings_qty')
        if qty_span:
            quantity = int(qty_span['data-qty'])
        else:
            quantity = None

        price_span = div.find('span', class_='normal_price', attrs={'data-price': True})
        if price_span:
            price = int(price_span['data-price']) / 100
        else:
            price = None

        name_span = div.find('span', class_='market_listing_item_name')
        if name_span:
            name = name_span.text
        else:
            name = None

        wear = None
        if name and '(' in name and name.endswith(')'):
            wear = name[name.rfind('(') + 1:-1]

        item = {
            'name': name,
            'hash_name': hash_name,
            'wear': wear,
            'url': url,
            'price_usd': price,
            'quantity': quantity,
            'scraped_at': datetime.now().isoformat(),
        }
        items.append(item)

    return items


def scrape_all():
    first = fetch_page(0)
    if first is None:
        print("Erste Seite konnte nicht geladen werden, breche ab.")
        return []

    total_count = first['total_count']
    print(f"Total zu sammeln: {total_count}")

    all_items = parse_page(first['results_html'])
    start = len(all_items)

    while start < total_count:
        time.sleep(DELAY)
        page = fetch_page(start)

        if page is None:
            print(f"Seite ab start={start} nicht ladbar, breche ab.")
            break

        page_items = parse_page(page['results_html'])

        if not page_items:
            print(f"Keine weiteren Items ab start={start}, breche ab.")
            break

        all_items.extend(page_items)
        start = start + len(page_items)
        print(f"Gesammelt {len(all_items)} / {total_count}")

    return all_items


if __name__ == '__main__':
    data = scrape_all()

    fieldnames = ['name', 'hash_name', 'wear', 'url', 'price_usd', 'quantity', 'scraped_at']
    with open('1-raw_data.csv', 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(data)
    print(f"{len(data)} Items in 1-raw_data.csv gespeichert")

Total zu sammeln: 570
Gesammelt 20 / 570
Gesammelt 30 / 570
Gesammelt 40 / 570
Gesammelt 50 / 570
Gesammelt 60 / 570
Gesammelt 70 / 570
Gesammelt 80 / 570
Gesammelt 90 / 570
Gesammelt 100 / 570
Gesammelt 110 / 570
Gesammelt 120 / 570
Gesammelt 130 / 570
Gesammelt 140 / 570
Gesammelt 150 / 570
Gesammelt 160 / 570
Gesammelt 170 / 570
Gesammelt 180 / 570
Gesammelt 190 / 570
Gesammelt 200 / 570
Gesammelt 210 / 570
Gesammelt 220 / 570
Gesammelt 230 / 570
Gesammelt 240 / 570
Gesammelt 250 / 570
Gesammelt 260 / 570
Gesammelt 270 / 570
Gesammelt 280 / 570
Gesammelt 290 / 570
Gesammelt 300 / 570
Gesammelt 310 / 570
Gesammelt 320 / 570
Gesammelt 330 / 570
Gesammelt 340 / 570
Gesammelt 350 / 570
Gesammelt 360 / 570
Gesammelt 370 / 570
Gesammelt 380 / 570
Gesammelt 390 / 570
Gesammelt 400 / 570
Gesammelt 410 / 570
Gesammelt 420 / 570
Versuch 1 fehlgeschlagen bei start=420: 429 Client Error: Too Many Requests for url: https://steamcommunity.com/market/search/render/?appid=730&category_730_Weapon%5B